In [1]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

In [2]:
import hopsworks

In [3]:
import os
from dotenv import dotenv_values
ENV = dotenv_values("./work/AIEngineering/machine_learning/feature_store/hopsworks/.env")

In [37]:
project = hopsworks.login(
    project='my_first_project10',  
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=ENV['api_key'],
)

2026-09-04 13:35:40,505 INFO: Closing external client and cleaning up certificates.
2026-09-04 13:35:40,510 INFO: Connection closed.
2026-09-04 13:35:40,513 INFO: Initializing external client
2026-09-04 13:35:40,513 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-04 13:35:43,808 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42213


In [38]:
fs = project.get_feature_store()

In [13]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_spark_hopsworks').getOrCreate()

In [25]:
spark = SparkSession.builder \
    .appName("Hopsworks-Spark-App") \
    .config("spark.jars.packages", "com.logicalclocks:hsfs-spark-3.3_2.12:3.7.0") \
    .getOrCreate()

In [13]:
import pandas as pd
data = {
    "pessoa_id": [1001, 1002, 1003, 1004], 
    "idade": [28, 45, 32, 19],
    "gastos_30d": [1250.50, 4300.00, 890.20, 150.00],
    "media_saldo_6m": [3200.00, 12500.00, 2100.00, 400.00],
    "dias_desde_ultimo_login": [1, 5, 2, 0]
}

df_pessoa = pd.DataFrame(data)

In [22]:
import pandas as pd
from datetime import date
data = {
    "pessoa_id": [1001, 1002, 1003, 1004],
    "peso": [110, 68, 90, 62],
    "time_futebol": ["palmeiras", "flamengo", "cruzeiro", "vasco"],
    "id_pais": ["Brasil", "Japao", "Espanha", "Brasil"],
    "quantos_dias_sem_comer": [1, 2, 4, 3],
}

df_identidade = pd.DataFrame(data)

df_financeiro = pd.DataFrame({
    "pessoa_id": [1001, 1002, 1003, 1001],
    "gastos_100d": [1200.0, 8500.0, 400.0, 2000.0],
    "inadimplente": [0, 0, 1, 0], # Nosso Alvo (Target)
    "timestamp": ['2025-03', '2026-01', '2022-01', '2025-05']
})

dados_spark = [[1001,'comfy', date(2025,3,1)],[1002, 'flexform', date(2025,4,1)],[1003, 'dt3', date(2024,4,1)], [1001,'thunderx', date(2024,4,1)]]
df_comportamento_cadeira = spark.createDataFrame(dados_spark, schema='pessoa_id int, cadeira string, timestamp date')

In [ ]:
pessoa_fg = fs.get_or_create_feature_group(
    name="pessoa_comportamento_financeiro",
    version=2,
    primary_key=["pessoa_id"],
    description="Comportamento financeiro e de engajamento da entidade Pessoa",
    online_enabled=True # Habilita acesso de baixa latência para predições em tempo real
)

pessoa_fg.insert(df_pessoa)

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/42213/fs/30898/fg/52449


Uploading Dataframe: 100.00% |██████████| Rows 4/4 | Elapsed Time: 00:02 | Remaining Time: 00:00


(None, None)

In [ ]:
fg_identidade = fs.get_or_create_feature_group(
    name="pessoa_comportamento_identitario",
    version=2,
    primary_key=["pessoa_id"],
    online_enabled=True
)
fg_identidade.insert(df_identidade)

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/42213/fs/30898/fg/52450


Uploading Dataframe: 100.00% |██████████| Rows 4/4 | Elapsed Time: 00:01 | Remaining Time: 00:00


(None, None)

In [ ]:
fg_financeiro = fs.get_or_create_feature_group(
    name="pessoa_comportamento_financeiro_plus",
    version=2,
    primary_key=["pessoa_id"],
    online_enabled=True
)
fg_financeiro.insert(df_financeiro)

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/42213/fs/30898/fg/52451


Uploading Dataframe: 100.00% |██████████| Rows 4/4 | Elapsed Time: 00:01 | Remaining Time: 00:00


(None, None)

In [30]:
import pandas as pd

df_pandas = df_comportamento_cadeira.toPandas()
df_pandas["timestamp"] = pd.to_datetime(df_pandas["timestamp"])

# Increment the version to automatically map the new schema
fg_cadeira_v4 = fs.get_or_create_feature_group(
    name="pessoa_comportamento_cadeira",
    version=4,
    primary_key=["pessoa_id"],
    event_time="timestamp",
    online_enabled=True
)

fg_cadeira_v4.insert(df_pandas)

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/42213/fs/30898/fg/54290


Uploading Dataframe: 100.00% |██████████| Rows 4/4 | Elapsed Time: 00:02 | Remaining Time: 00:00


(None, None)

In [6]:
all_fgs = fs.get_feature_groups()

In [7]:
for fgs in all_fgs:
    fgs.name

'pessoa_comportamento_financeiro'

'pessoa_comportamento_identitario'

'pessoa_comportamento_financeiro_plus'

'pessoa_comportamento_financeiro_plus'

'pessoa_comportamento_financeiro'

'pessoa_comportamento_identitario'

'pessoa_comportamento_cadeira'

In [8]:
a = fs.get_feature_group(name="pessoa_comportamento_cadeira", version=2)

In [10]:
a.read(online=True)

,pessoa_id,cadeira,timestamp
0,1002,flexform,2025-01
1,1003,dt3,2024-04
2,1001,thunderx,2024-04


In [15]:
fs.get_feature_group(name="pessoa_comportamento_identitario", version=2).read()

2026-09-04 12:03:38,237 ERROR: Flight returned unavailable error, with message: Socket closed. gRPC client debug context: UNKNOWN:Error received from peer ipv4:57.130.64.132:5005 {grpc_status:14, grpc_message:"Socket closed"}. Client context: IOError: Server never sent a data message. Detail: Internal
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/hsfs/core/arrow_flight_client.py", line 433, in afs_error_handler_wrapper
    return func(instance, *args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/hsfs/core/arrow_flight_client.py", line 505, in _read_query
    return self._get_dataset(
           ^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/retrying.py", line 55, in wrapped_f
    return Retrying(*dargs, **dkw).call(f, *args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/retrying.py", line 289, in call
    raise attempt.

Error: Reading data from Hopsworks, using Hopsworks Feature Query Service           


FeatureStoreException: Could not read data using Hopsworks Query Service.

In [ ]:
fg_financeiro_plus.read()

In [5]:
# Example: Read an existing feature group
fg_comportamento_financeiro = fs.get_feature_group(
    name="pessoa_comportamento_financeiro",
    version=2
)


fg_comportamento_identitario = fs.get_feature_group(
    name="pessoa_comportamento_identitario",
    version=2
)

fg_financeiro_plus = fs.get_feature_group(
    name="pessoa_comportamento_financeiro_plus",
    version=2
)

In [32]:
for i in fs.get_feature_groups():
    i.name

'pessoa_comportamento_financeiro_plus'

'pessoa_comportamento_financeiro'

'pessoa_comportamento_identitario'

'pessoa_comportamento_financeiro_plus'

'pessoa_comportamento_cadeira'

'pessoa_comportamento_financeiro'

'pessoa_comportamento_identitario'

'pessoa_comportamento_cadeira'

'pessoa_comportamento_cadeira'

In [39]:
fg = fs.get_feature_group('pessoa_comportamento_cadeira', version=4)

In [ ]:
fg.read()

In [36]:
fg.read(online=True, dataframe_type="polars")

ModuleNotFoundError: Polars package not found. If you want to use Polars with Hopsworks you can install the corresponding extra via '`pip install "hopsworks[polars]"`. 'You can also install polars directly in your environment with `pip install polars`. You will need to restart your kernel if applicable.

In [10]:
fg_comportamento_financeiro.read()

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.75s) 


,pessoa_id,idade,gastos_30d,media_saldo_6m,dias_desde_ultimo_login
0,1001,28,1250.5,3200.0,1
1,1002,45,4300.0,12500.0,5
2,1003,32,890.2,2100.0,2
3,1004,19,150.0,400.0,0


In [ ]:
# Read data from the feature group
df = fg.read()
print(f"Connected to {project.name}. Feature group has {len(df)} rows")

# For model serving
ms = project.get_model_serving()

# For model registry
mr = project.get_model_registry()

##  Feature View

In [ ]:
all_fgs = fs.get_feature_groups()

In [ ]:
for fg in all_fgs:
  print(f"Nome: {fg.name}, Versão: {fg.version}")

Nome: pessoa_comportamento_financeiro, Versão: 1
Nome: pessoa_comportamento_identitario, Versão: 1
Nome: pessoa_comportamento_financeiro_plus, Versão: 1


In [ ]:
query_features = fg_comportamento_financeiro.select(["pessoa_id", "idade", "dias_desde_ultimo_login"]) \
    .join(fg_financeiro_plus.select(["gastos_100d", "inadimplente", "timestamp"])) \
        .join(fg_comportamento_identitario.select(["peso", "quantos_dias_sem_comer"]))

In [ ]:
feature_view = fs.create_feature_view(
    name="modelo_forecasting",
    version=3,
    labels=["inadimplente"],
    query=query_features
)

Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/42213/fs/30898/fv/modelo_forecasting/version/3


In [ ]:
feature_view = fs.get_feature_view(name="modelo_forecasting", version=3)

In [ ]:
feature_vector = feature_view.get_feature_vector({"pessoa_id": 1001})

In [ ]:
feature_vector

[1001, 28, 1, 2000.0, '2025-05', 110, 1]

### Metodo online

In [ ]:
feature_vectors = feature_view.get_feature_vectors(
    entry=[{"pessoa_id": 1001}, {"pessoa_id": 1002}, {"pessoa_id":1001}]
)
feature_vectors

[[1001, 28, 1, 2000.0, '2025-05', 110, 1],
 [1002, 45, 5, 8500.0, '2026-01', 68, 2],
 [1001, 28, 1, 2000.0, '2025-05', 110, 1]]

### Metodo offline (batch) training dataset

In [ ]:
X_train, X_test, y_train, y_test = feature_view.train_test_split(
    test_size=0.2,
    description="Dataset para treino do modelo de risco v1"
)

2026-09-03 16:59:52,220 INFO: Computing insert statistics
2026-09-03 16:59:52,232 INFO: Computing insert statistics


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (3.58s) 


In [ ]:
X_train

,pessoa_id,idade,dias_desde_ultimo_login,gastos_100d,timestamp,peso,quantos_dias_sem_comer
0,1001,28,1,1200.0,2025-03,110,1
1,1002,45,5,8500.0,2026-01,68,2
3,1001,28,1,2000.0,2025-05,110,1
4,1004,19,0,NaN,None,62,3


In [ ]:
df
feature_vectors = feature_view.get_feature_vectors(
    entry=[{"pessoa_id": 1001}, {"pessoa_id": 1002}, {"pessoa_id":1001}],
    passed_features=[{"idade":30}, {"idade":33}, {"idade":46}]
)
feature_vectors

,pessoa_id,idade,dias_desde_ultimo_login,gastos_100d,peso,quantos_dias_sem_comer
0,1001,28,1,1200.0,110,1
1,1002,45,5,8500.0,68,2
2,1003,32,2,400.0,90,4
3,1004,19,0,NaN,62,3


[[1003, 30, 2, 400.0, 90, 4],
 [1002, 33, 5, 8500.0, 68, 2],
 [1001, 46, 1, 1200.0, 110, 1]]

In [ ]:
feature_vectors

[[32, 2, 400.0, 90, 4], [45, 5, 8500.0, 68, 2]]

In [ ]:
# Suponha que você tenha uma tabela de eventos (ex: compras da pessoa)
# com uma coluna 'event_timestamp'

# 1. Definindo um filtro de data de corte fixo na query
query = fg_cadastral.select_all() \
    .join(fg_financeiro.select_all())

# 2. Buscando dados historicos no OFFLINE usando filtro de data
df_treino_historico = query.filter(
    fg_financeiro.event_timestamp < "2023-12-31 23:59:59"
).read()